## Imports

In [1]:
import sys
import os

# Agregar carpeta raíz del proyecto

sys.path.append(
    os.path.abspath("..")
)

## Carga de datos

In [2]:
from src.cargar_datos import cargar_datos

personal, calendario = cargar_datos(
    "../data/cerebro_farallones.xlsx"
)

## Crear turnos

Aqui se convierte el calendario en la demanda operativa, genera automaticamente todos los turnos del mes

In [3]:
from src.crear_turnos import crear_turnos
turnos_df = crear_turnos(calendario)

## Crear grupos

In [4]:
from src.grupos import crear_grupos

grupos = crear_grupos(personal)

pvc = grupos["pvc"]

no_pvc = grupos["no_pvc"]

conductores_carro = grupos[
    "conductores_carro"
]

conductores_moto = grupos[
    "conductores_moto"
]

ecoturismo = grupos[
    "ecoturismo"
]

anchicaya = grupos[
    "anchicaya"
]

sin_arbolito = grupos[
    "sin_arbolito"
]

no_pvc_restringidos = grupos[
    "no_pvc_restringidos"
]

## Crear modelo

In [5]:
from ortools.sat.python import cp_model

# =========================
# CREAR MODELO
# =========================

model = cp_model.CpModel()

# =========================
# VARIABLES DE DECISIÓN
# =========================

x = {}

for p in personal.index:

    for t in turnos_df.index:

        x[(p, t)] = model.NewBoolVar(
            f"x_{p}_{t}"
        )

## Restricciones

- cobertura = Cada puesto tiene el numero de personas idoneas
- elegibilidad = Solo personal activo y que pueda hacer el puesto
- no doble turno = Nadie puede hacer 2 turnos en un dia
- Transporte = Arbolito noche prioridad carro, arbolito dia moto-carro, pato pance carro o dos motos
- Ecoturismo = Al menos 1 de ecoturismo en pato-pance
- Pato = No mas de 2 puestos seguidos
- Anchicaya = Los de anchicayá solo 1 puesto arbolito noche a la semana


In [6]:
from src.restricciones import (
    agregar_restriccion_cobertura,
    agregar_restriccion_elegibilidad,
    agregar_restriccion_no_doble_turno,
    agregar_restriccion_transporte,
    agregar_restriccion_ecoturismo,
    agregar_restriccion_patitos,
    agregar_restriccion_anchicaya
)

turnos_pato = turnos_df[
    turnos_df["puesto"] == "Pato Pance"
].index.tolist()


agregar_restriccion_cobertura(
    model,
    x,
    personal,
    turnos_df
)

agregar_restriccion_elegibilidad(
    model,
    x,
    personal,
    turnos_df
)

agregar_restriccion_no_doble_turno(
    model,
    x,
    personal,
    turnos_df
)

agregar_restriccion_transporte(

    model,
    x,
    turnos_df,
    conductores_carro,
    conductores_moto

)

agregar_restriccion_ecoturismo(

    model,
    x,
    turnos_df,
    ecoturismo

)

agregar_restriccion_patitos(

    model,
    x,
    personal,
    turnos_pato

)

agregar_restriccion_anchicaya(

    model,
    x,
    personal,
    turnos_df,
    anchicaya

)

## Objetivo

In [7]:
from src.objetivo import (
    agregar_balance_carga,
    agregar_funcion_objetivo
)

turnos_arbolito = turnos_df[
    turnos_df["puesto"] == "Arbolito"
].index.tolist()

turnos_pato = turnos_df[
    turnos_df["puesto"] == "Pato Pance"
].index.tolist()

agregar_balance_carga(

    model,
    x,
    personal,
    turnos_df,
    turnos_arbolito,
    turnos_pato,
    pvc,
    no_pvc,
    no_pvc_restringidos,
    sin_arbolito

)

agregar_funcion_objetivo(

    model,
    x,
    personal,
    turnos_df,
    no_pvc

)

## Solver

In [8]:
# =========================
# SOLVER
# =========================

solver = cp_model.CpSolver()

status = solver.Solve(model)

## Reportes

In [9]:
from src.reportes import (
    imprimir_asignaciones,
    crear_resumen_carga
)
from src.exportar import (
    exportar_resultados
)

if status == cp_model.FEASIBLE or status == cp_model.OPTIMAL:

    imprimir_asignaciones(

        solver,
        x,
        personal,
        turnos_df

    )

    resumen_df = crear_resumen_carga(

        solver,
        x,
        personal,
        turnos_df

    )

    display(resumen_df)


    exportar_resultados(

        solver,
        x,
        personal,
        turnos_df,
        resumen_df,

        "../outputs/resultados_turnos.xlsx"

    )
    
else:

    print(
        "No se encontró solución."
    )


=== ASIGNACIONES ===

2026-06-01 00:00:00 | Arbolito | dia
- Diana Murillo
- Eider Montaño
(Fmu-16G)

2026-06-01 00:00:00 | Arbolito | noche
- Alexander Morales
- Carlos Perea
- Juan Manuel Guzman
(Ojz-741)

2026-06-02 00:00:00 | Arbolito | dia
- Hernan Montoya
- Rafael Pardo

2026-06-02 00:00:00 | Arbolito | noche
- Alexander Morales
- Cristian Libreros 
(Okz-055)
- Diana Murillo

2026-06-03 00:00:00 | Arbolito | dia
- Maria Fernanda Parra
- Rafael Pardo

2026-06-03 00:00:00 | Arbolito | noche
- Dannythza Mona
- John Cobaleda
- Oscar Quiñones

2026-06-04 00:00:00 | Arbolito | dia
- Eider Montaño
(Fmu-16G)
- Miguel Castro

2026-06-04 00:00:00 | Arbolito | noche
- Alejandro Nuñez
(Tyn-42F)
- Carlos Perea
- Cesar Rueda
(Fly-30G)

2026-06-05 00:00:00 | Arbolito | dia
- Alejandro Nuñez
(Tyn-42F)
- Jhon Freider Troches

2026-06-05 00:00:00 | Arbolito | noche
- Eider Montaño
(Fmu-16G)
- Gustavo Rodríguez
- Luís Carlos Mamian
(Okz057)

2026-06-06 00:00:00 | Arbolito | dia
- Alejandro Nuñez
(

,nombre,estrategia,conductor,arbolito_dia,arbolito_noche,pato_pance,total
0,Alejandra Garcia,PVC,MOTO,4,1,0,5
1,Alejandro Nuñez\n(Tyn-42F),FUNCIONARIO,CARRO y MOTO,3,2,0,5
2,Alexander Gomez\n(Fly-32G),ECOTURISMO,CARRO y MOTO,0,2,3,5
4,Alfredo Abadia\n(Fly 34G),FUNCIONARIO,CARRO y MOTO,1,4,0,5
10,Camilo Castañeda,PVC,NO,1,4,0,5
...,...,...,...,...,...,...,...
48,Luz Dalia Miranda,ANCHICAYA,NO,0,0,0,0
49,Luz Ester Restrepo,ANCHICAYA,MOTO,0,0,0,0
62,Sebastian Ovalle,MONITOREO,NO,0,0,0,0
63,Sharon Becerra,RH,NO,0,0,0,0



Archivo exportado:
../outputs/resultados_turnos.xlsx
